# 05 | RAG中的JSON文档加载（JSONLoader）

上一篇看的是 `CSVLoader`。CSV 的结构比较平：一行就是一条记录。

JSON 不太一样。它经常是嵌套的：外面有导出信息，里面有数组，数组里又有一条条业务数据。

`JSONLoader` 要解决的问题就是：**从 JSON 里挑出真正要进入 RAG 的那部分内容，然后变成 LangChain 的 `Document`。**

它不是让模型读懂 JSON 的魔法棒，更像一把小镊子：JSON 很大，但你要夹出来的是能被检索的正文。

## 一、先把位置摆清楚

一条典型 RAG 流程还是这样：

```text
JSON / CSV / PDF / 网页
-> Document loader
-> Document
-> Text splitter
-> Embedding
-> Vector store
-> Retriever
-> LLM 回答
```

`JSONLoader` 只负责前面这一步：

```text
本地 JSON 文件 -> 一组 Document
```

官方文档里的关键点是：`JSONLoader` 用 `jq_schema` 从 JSON 中选择内容。也就是说，JSONLoader 的重点不是“能不能读 JSON”，而是“你到底想从 JSON 的哪个位置读”。

## 二、场景：售后知识库 JSON

假设一个电商团队有一批售后知识库文章，比如退款、发票、会员续费、质量问题。

这些文章不是存在 CSV 里，而是从 CMS 或后台接口导出成 JSON。

业务同事想问：

```text
未发货订单能退款吗？
发票在哪里下载？
自动续费怎么关闭？
```

如果要做 RAG，第一步还是一样：先把 JSON 里的每篇文章变成 `Document`。

这篇就用这个场景。够业务，也不会一上来把人扔进一坨无名 JSON。

## 三、准备一份本地 JSON

为了让 notebook 可以自己跑，先生成一份小 JSON。

注意它不是一个简单数组，而是更接近真实导出结果：外层有导出时间、系统名，真正的文章在 `articles` 数组里。

In [ ]:
import json
from pathlib import Path

# 统一把示例数据放到 rag/data 目录。
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

json_path = data_dir / "support_articles.json"

# 一个接近真实后台导出的 JSON。
# 注意：真正要进入 RAG 的文章不在最外层，而是在 articles 数组里。
# 这也是 JSONLoader 需要 jq_schema 的原因：JSON 往往不是平铺结构。
support_data = {
    "exported_at": "2026-06-01T09:00:00+08:00",
    "source_system": "support-cms",
    "articles": [
        {
            # article_id 类似知识库文章编号，后面可以放进 metadata 用来追来源。
            "article_id": "KB-1001",
            "title": "未发货订单如何退款",
            "category": "退款政策",
            # content 是真正希望进入 RAG 检索的正文。
            "content": "如果订单还没有发货，用户可以在订单详情页申请退款，通常会在 1 到 3 个工作日内原路退回。",
            "updated_at": "2026-05-20",
        },
        {
            "article_id": "KB-1002",
            "title": "发票在哪里下载",
            "category": "发票服务",
            "content": "用户可以进入账户中心的发票管理页面，选择对应订单后下载电子发票。",
            "updated_at": "2026-05-21",
        },
        {
            "article_id": "KB-1003",
            "title": "如何关闭自动续费",
            "category": "会员服务",
            "content": "用户可以在会员中心关闭自动续费，关闭后当前会员权益仍然保留到到期日。",
            "updated_at": "2026-05-22",
        },
    ],
}

# 写成本地 JSON 文件，模拟从 CMS / 知识库后台导出的数据。
json_path.write_text(
    json.dumps(support_data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json_path)

## 四、先用 `jq_schema` 取正文

使用 `JSONLoader` 前，需要安装两个包：

```bash
uv add langchain-community jq
```

`jq` 是这里的重点。`jq_schema` 写的就是 jq 表达式，用来告诉 loader：你要从 JSON 的哪个位置拿内容。

比如：

```text
.articles[].content
```

意思是：进入 `articles` 数组，遍历每一篇文章，取出里面的 `content` 字段。

In [ ]:
from langchain_community.document_loaders import JSONLoader

# 方式一：直接取文章正文。
# .articles[] 表示遍历 articles 数组里的每一篇文章。
# .content 表示只取每篇文章里的 content 字段。
loader = JSONLoader(
    file_path=str(json_path),
    jq_schema=".articles[].content",
)

# load() 会一次性加载所有匹配到的内容。
# 这里会得到 3 个 Document，因为 articles 里有 3 篇文章。
documents = loader.load()

print(len(documents))
print(documents[0].page_content)
print(documents[0].metadata)

这样已经能得到 `Document` 了。

但它有一个问题：`page_content` 里只有正文，标题、分类、文章编号都没进来。

对简单 Demo 来说没关系；对真实 RAG 来说，有点可惜。因为用户问“自动续费怎么关”，你不只想检索到正文，还想知道这条内容来自哪篇知识库文章。

## 五、用 `content_key` 指定正文

更常见的写法是：先用 `jq_schema` 选中一条完整记录，再用 `content_key` 指定哪一个字段作为正文。

也就是：

```text
jq_schema='.articles[]'  ->  选中每篇文章
content_key='content'   ->  把文章里的 content 当作 page_content
```

这样后面就有机会把 `article_id`、`title`、`category` 放进 metadata。

In [ ]:
# 方式二：先选中整篇文章，再指定 content 字段作为正文。
# jq_schema=".articles[]"：每个 Document 对应 articles 里的一个完整对象。
# content_key="content"：从这个对象里拿 content 字段作为 page_content。
loader = JSONLoader(
    file_path=str(json_path),
    jq_schema=".articles[]",
    content_key="content",
)

documents = loader.load()

print(documents[0].page_content)
print(documents[0].metadata)

这一步读起来会更像真实项目。

`jq_schema` 负责“找到每篇文章”，`content_key` 负责“拿哪一列当正文”。

如果把 JSON 想成一个仓库：`jq_schema` 是走到哪个货架，`content_key` 是从货架上拿哪一格。这个比喻有点土，但确实好记。

## 六、用 `metadata_func` 保留业务信息

现在还有一个关键问题：metadata 里默认只有文件来源和序号。

但真实 RAG 里，我们通常希望保留这些信息：

- `article_id`：来自哪篇知识库文章
- `title`：文章标题
- `category`：退款、发票、会员服务等分类
- `updated_at`：这条知识什么时候更新的

这时就用 `metadata_func`。它会接收当前 JSON 记录和默认 metadata，然后返回更新后的 metadata。

In [ ]:
def metadata_func(record: dict, metadata: dict) -> dict:
    """把 JSON 记录里的业务字段放进 Document metadata。"""
    # record 是 jq_schema=".articles[]" 选中的那一篇文章。
    # metadata 是 JSONLoader 默认生成的 metadata，比如 source、seq_num。
    # 这里是在默认 metadata 的基础上，补充业务上更有用的信息。
    metadata["article_id"] = record.get("article_id")
    metadata["title"] = record.get("title")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")
    return metadata


loader_with_metadata = JSONLoader(
    file_path=str(json_path),
    # 先选中每一篇完整文章，这样 metadata_func 才能拿到 title、category 等字段。
    jq_schema=".articles[]",
    # 只有 content 字段进入 page_content，避免把整篇 JSON 原样塞给模型。
    content_key="content",
    # 把 article_id、title、category、updated_at 补到 metadata 里。
    metadata_func=metadata_func,
)

documents_with_metadata = loader_with_metadata.load()

print(documents_with_metadata[0].page_content)
print(documents_with_metadata[0].metadata)

`metadata_func` 的业务意义很直接。

没有它，你只知道：

```text
这条内容来自 support_articles.json
```

有了它，你能知道：

```text
这条内容来自 KB-1001：《未发货订单如何退款》
分类：退款政策
更新时间：2026-05-20
```

这就不只是“能搜到”，而是“搜到以后还能解释来源”。RAG 里这件事很重要，不然模型回答得再像真的，也可能让人心里没底。

## 七、大文件时用 `lazy_load()`

`load()` 会一次性加载所有 Document。

如果 JSON 很大，比如几万篇知识库文章、几十万条日志，就可以用 `lazy_load()` 一条一条处理。

In [ ]:
# lazy_load() 返回迭代器，不会一次性把所有 Document 都放进内存。
# 如果后面要分批写入向量库，这种方式会更稳。
# 这里为了演示，只取前 2 条。
for index, document in enumerate(loader_with_metadata.lazy_load()):
    print(document.metadata)
    print(document.page_content)
    print("-" * 40)

    if index >= 1:
        break

## 八、JSON Lines 也顺手跑一下

官方文档还提到一种常见格式：JSON Lines，也就是 `.jsonl`。

普通 JSON 经常长这样：

```json
[
  {"id": "1", "text": "第一条"},
  {"id": "2", "text": "第二条"}
]
```

JSON Lines 则是这样：

```jsonl
{"id": "1", "text": "第一条"}
{"id": "2", "text": "第二条"}
```

也就是：**一行一个 JSON 对象，完了还没有逗号分割**。

这种格式很适合日志、埋点、客服对话、模型调用记录。因为新数据来了以后，直接往文件末尾追加一行就行，不用重新维护一个完整的大数组。朴素，但工程上很好用。


In [ ]:
# 准备一份 JSON Lines 示例文件。
# JSON Lines 的特点是：一行就是一个独立 JSON 对象。
# 这里每一行代表一条客服对话摘要。
jsonl_path = data_dir / "support_chat_logs.jsonl"

jsonl_records = [
    {
        "conversation_id": "C-2001",
        "topic": "退款",
        "summary": "用户询问未发货订单是否可以退款，客服说明可在订单详情页申请。",
    },
    {
        "conversation_id": "C-2002",
        "topic": "发票",
        "summary": "用户询问电子发票下载入口，客服引导到发票管理页面。",
    },
    {
        "conversation_id": "C-2003",
        "topic": "会员续费",
        "summary": "用户询问如何关闭自动续费，客服说明可在会员中心处理。",
    },
]

# 注意这里不是 json.dumps(jsonl_records)。
# JSON Lines 要逐条 dumps，再用换行符拼起来。
jsonl_path.write_text(
    "\n".join(json.dumps(record, ensure_ascii=False) for record in jsonl_records),
    encoding="utf-8",
)

print(jsonl_path)

加载 JSON Lines 时，要加 `json_lines=True`。

因为现在不是“一个 JSON 文件里有一个大对象”，而是“每一行都是一个 JSON 对象”。loader 需要按行解析。

In [ ]:
def chat_log_metadata_func(record: dict, metadata: dict) -> dict:
    """把对话 id 和主题放进 metadata，方便后续追来源。"""
    # 对 JSON Lines 来说，record 就是当前这一行的 JSON 对象。
    # 这里把 conversation_id 覆盖到 source，后面更容易知道内容来自哪段对话。
    metadata["source"] = record.get("conversation_id")
    metadata["topic"] = record.get("topic")
    return metadata


jsonl_loader = JSONLoader(
    file_path=str(jsonl_path),
    # JSON Lines 每一行就是一个完整对象，所以用 . 选中当前行。
    jq_schema=".",
    # 每行里的 summary 字段作为 Document 正文。
    content_key="summary",
    metadata_func=chat_log_metadata_func,
    # 关键参数：告诉 JSONLoader 这个文件是一行一个 JSON 对象。
    json_lines=True,
)

jsonl_documents = jsonl_loader.load()

print(len(jsonl_documents))
print(jsonl_documents[0].page_content)
print(jsonl_documents[0].metadata)

这里的 `jq_schema="."` 表示：每一行本身就是一条记录，直接选中当前对象。

然后 `content_key="summary"` 表示：把这条记录里的 `summary` 当作正文。

所以 JSON Lines 这类数据，读法可以这样记：

```text
json_lines=True：按行读 JSON
jq_schema="."：每一行就是当前记录
content_key="summary"：summary 字段当正文
metadata_func：把 conversation_id、topic 这类业务字段放进 metadata
```

如果以后要做“客服对话记录 RAG”“模型调用日志检索”“用户行为日志分析”，JSON Lines 会比一个巨大的 JSON 数组更顺手。大数组一旦很大，编辑和追加都会让人开始怀疑人生。

## 九、加载之后下一步是什么

现在我们拿到的是 `Document`，RAG 的第一步完成了。

后面通常会继续做：

```text
Document
-> 切分 / 清洗
-> Embedding
-> 写入向量库
-> 检索相关文章
-> 交给 LLM 组织回答
```

最后我会这么记：

```text
JSONLoader：把 JSON 里的指定节点变成 Document
jq_schema：从 JSON 里选哪一批记录
content_key：每条记录里哪个字段当正文
metadata_func：把业务字段塞进 metadata，方便追来源
lazy_load：大文件时别一口气全吞
json_lines=True：按 JSON Lines 格式逐行解析
```

JSONLoader 的难点不是加载文件，而是选对位置。位置选错，后面 RAG 就会很努力地处理一堆不该处理的东西。